# 14 — Reviewer Experiments (A2/A3/A5/B5)

**Jalankan di SageMaker SETELAH data CIC (`cleaned_100.pkl`) + UNSW CSV tersedia.**
Notebook ini menambah bukti kuantitatif yang diminta reviewer:

- **A2** repeated-seed (mean/std/95% CI) untuk MCC clean & adversarial.
- **A3** serangan lebih kuat: **PGD iteratif** functional-preserving (bukan hanya FGSM 1-langkah).
- **A5** **Attack Success Rate (ASR)** + metrik berbasis confusion-matrix (recall/precision/F1) per kondisi.
- **B5** **ablation komponen-level**: {baseline, +few-shot, +adv, +few-shot+adv} x {clean, PGD}.

Hasil ditulis ke `paper2_reviewer_out/` + diupload ke S3 `unsw-far/paper2_reviewer/`.
Angka NYATA dari notebook ini yang nanti dimasukkan ke paper (Tabel A2/A3/A5/B5).

## 0. Setup

In [ ]:
import importlib, subprocess, sys
for pkg,imp in [('pandas','pandas'),('numpy','numpy'),('scikit-learn','sklearn'),('xgboost','xgboost'),('scipy','scipy'),('boto3','boto3')]:
    try: importlib.import_module(imp)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
import os, json, pickle
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import matthews_corrcoef, f1_score, precision_score, recall_score, confusion_matrix
from xgboost import XGBClassifier
from scipy import stats

CIC_PKL='../../CICDDoS2018/data/cleaned_100.pkl'
UNSW_TRAIN='../data/UNSW_NB15_testing-set.csv'; UNSW_TEST='../data/UNSW_NB15_training-set.csv'
OUTDIR='paper2_reviewer_out'; os.makedirs(OUTDIR,exist_ok=True)
S3_BUCKET=os.environ.get('S3_BUCKET','ssh-detection-features-232032302717'); REGION='ap-southeast-1'
SEEDS=[13,42,101,202,303]          # A2: repeated seeds
H=0.01; EPS_TRAIN=0.1; ADV_RATIO=0.20; FEWSHOT_FRAC=0.01; MAXN=40000
EPS_EVAL=[0.05,0.1,0.2]; PGD_ITERS=10; PGD_ALPHA=0.02   # A3: PGD
print('files:', os.path.exists(CIC_PKL), os.path.exists(UNSW_TRAIN), os.path.exists(UNSW_TEST))

## 1. SFM mapping + util (konsisten nb11/12)

In [ ]:
MAP_A={'duration':('Flow Duration','dur'),'fwd_pkts':('Tot Fwd Pkts','spkts'),
       'bwd_pkts':('Tot Bwd Pkts','dpkts'),'fwd_bytes':('TotLen Fwd Pkts','sbytes'),
       'bwd_bytes':('TotLen Bwd Pkts','dbytes'),'fwd_mean':('Fwd Pkt Len Mean','smean'),
       'bwd_mean':('Bwd Pkt Len Mean','dmean'),'src_load':('Flow Byts/s','sload'),
       'dst_load':('Bwd Pkts/s','dload')}
CANON=list(MAP_A.keys()); IX={c:i for i,c in enumerate(CANON)}
def build_matrix(df,side):
    idx=0 if side=='cic' else 1; cols=[MAP_A[c][idx] for c in CANON]
    out=df[cols].copy(); out.columns=CANON; out=out.replace([np.inf,-np.inf],np.nan)
    return out.fillna(out.median(numeric_only=True)).fillna(0.0).astype(float).values
def make_xgb(seed):
    return XGBClassifier(objective='binary:logistic',eval_metric='logloss',max_depth=8,
        learning_rate=0.1,n_estimators=200,subsample=0.8,colsample_bytree=0.8,
        n_jobs=-1,random_state=seed,tree_method='hist')
def loss_bin(model,X,y):
    p=np.clip(model.predict_proba(X)[:,1],1e-15,1-1e-15); y=y.astype(float)
    return -(y*np.log(p)+(1-y)*np.log(1-p))
def saliency(model,X,y,h=H):
    n,m=X.shape; S=np.zeros((n,m))
    for i in range(m):
        Xp=X.copy(); Xp[:,i]+=h; Xm=X.copy(); Xm[:,i]-=h
        S[:,i]=(loss_bin(model,Xp,y)-loss_bin(model,Xm,y))/(2*h)
    return S
def project_functional(Xs_scaled, mean, scale, X_ref_scaled=None):
    Xo=Xs_scaled*scale+mean; Xo=np.clip(Xo,0.0,None)
    Xo[:,IX['fwd_pkts']]=np.round(Xo[:,IX['fwd_pkts']]); Xo[:,IX['bwd_pkts']]=np.round(Xo[:,IX['bwd_pkts']])
    Xo[:,IX['fwd_bytes']]=np.maximum(Xo[:,IX['fwd_bytes']],Xo[:,IX['fwd_pkts']])
    Xo[:,IX['bwd_bytes']]=np.maximum(Xo[:,IX['bwd_bytes']],Xo[:,IX['bwd_pkts']])
    if X_ref_scaled is not None:            # monotonic add-only vs referensi asli
        Xr=X_ref_scaled*scale+mean
        for j in [IX[c] for c in ['fwd_pkts','bwd_pkts','fwd_bytes','bwd_bytes','duration']]:
            Xo[:,j]=np.maximum(Xo[:,j],Xr[:,j])
    with np.errstate(divide='ignore',invalid='ignore'):
        fm=np.where(Xo[:,IX['fwd_pkts']]>0,Xo[:,IX['fwd_bytes']]/Xo[:,IX['fwd_pkts']],0.0)
        bm=np.where(Xo[:,IX['bwd_pkts']]>0,Xo[:,IX['bwd_bytes']]/Xo[:,IX['bwd_pkts']],0.0)
    Xo[:,IX['fwd_mean']]=fm; Xo[:,IX['bwd_mean']]=bm
    # A4 fix: rekonsiliasi laju thd bytes/duration (perbaikan yang dijanjikan di paper)
    with np.errstate(divide='ignore',invalid='ignore'):
        d=np.where(Xo[:,IX['duration']]>0,Xo[:,IX['duration']],np.nan)
        Xo[:,IX['src_load']]=np.nan_to_num((Xo[:,IX['fwd_bytes']]+Xo[:,IX['bwd_bytes']])/d,nan=0.0)
        Xo[:,IX['dst_load']]=np.nan_to_num(Xo[:,IX['bwd_pkts']]/d,nan=0.0)
    return (Xo-mean)/scale
def fgsm_functional(X,S,eps,mean,scale):
    return project_functional(X+eps*np.sign(S),mean,scale,X_ref_scaled=X)
def pgd_functional(model,X,y,eps,mean,scale,iters=PGD_ITERS,alpha=PGD_ALPHA):
    # A3: PGD iteratif dgn proyeksi functional-preserving + clamp L-inf ke bola eps.
    Xadv=X.copy()
    for _ in range(iters):
        S=saliency(model,Xadv,y)
        Xadv=Xadv+alpha*np.sign(S)
        Xadv=np.clip(Xadv,X-eps,X+eps)              # L-inf ball
        Xadv=project_functional(Xadv,mean,scale,X_ref_scaled=X)
    return Xadv
def cm_metrics(y,yp):
    tn,fp,fn,tp=confusion_matrix(y,yp,labels=[0,1]).ravel()
    return dict(mcc=float(matthews_corrcoef(y,yp)),f1=float(f1_score(y,yp,zero_division=0)),
                recall=float(recall_score(y,yp,zero_division=0)),
                precision=float(precision_score(y,yp,zero_division=0)),
                tp=int(tp),fp=int(fp),fn=int(fn),tn=int(tn))
def asr(y,yp_clean,yp_adv):
    # A5: Attack Success Rate = fraksi attack yg SEMULA terdeteksi lalu LOLOS stlh serangan.
    atk=(y==1); det_before=atk & (yp_clean==1)
    if det_before.sum()==0: return float('nan')
    evaded=det_before & (yp_adv==0)
    return float(evaded.sum()/det_before.sum())
print('util siap')

## 2. Muat data + z-score (fit train saja)

In [ ]:
with open(CIC_PKL,'rb') as f: d=pickle.load(f)
cic_feats=list(d['feature_names']); X=np.asarray(d['X'],float); sc=d.get('scaler',None)
X_orig=X*sc.scale_+sc.mean_ if (sc is not None and hasattr(sc,'scale_')) else X
cic_df=pd.DataFrame(X_orig,columns=cic_feats)
benign=d.get('label_mapping',{}).get('Benign',0); y_cic=(np.asarray(d['y'])!=benign).astype(int)
unsw_tr=pd.read_csv(UNSW_TRAIN); unsw_te=pd.read_csv(UNSW_TEST)
y_utr=unsw_tr['label'].astype(int).values; y_ute=unsw_te['label'].astype(int).values
Xc_all=build_matrix(cic_df,'cic'); Xu_tr_raw=build_matrix(unsw_tr,'unsw'); Xu_te_raw=build_matrix(unsw_te,'unsw')
print('loaded CIC',Xc_all.shape,'UNSW tr/te',Xu_tr_raw.shape,Xu_te_raw.shape)

## 3. A2+B5: train 4 varian per seed, ukur clean & adversarial (FGSM+PGD)

Untuk tiap seed: split CIC (stratified), fit scaler train-only, latih 4 varian per arah,
lalu evaluasi clean-target + FGSM + PGD (adaptive white-box, functional-preserving).
Menghasilkan dataframe panjang untuk agregasi mean/std/CI.

In [ ]:
def train_variants(direction,X_src_tr,y_src_tr,X_tgt_tr,y_tgt_tr,seed):
    rng=np.random.RandomState(seed); M={}
    M['baseline']=make_xgb(seed).fit(X_src_tr,y_src_tr)
    nfs=max(1,int(len(X_tgt_tr)*FEWSHOT_FRAC)); ifs=rng.choice(len(X_tgt_tr),nfs,replace=False)
    Xfs=np.vstack([X_src_tr,X_tgt_tr[ifs]]); yfs=np.concatenate([y_src_tr,y_tgt_tr[ifs]])
    M['fewshot']=make_xgb(seed).fit(Xfs,yfs)
    def robust(base,Xtr,ytr):
        n=min(MAXN,len(Xtr)); idx=rng.choice(len(Xtr),n,replace=False)
        S=saliency(base,Xtr[idx],ytr[idx]); Xa=Xtr[idx]+EPS_TRAIN*np.sign(S)
        na=min(int(len(Xtr)*ADV_RATIO/(1-ADV_RATIO)),len(Xa)); sel=rng.choice(len(Xa),na,replace=False)
        return np.vstack([Xtr,Xa[sel]]),np.concatenate([ytr,ytr[idx][sel]])
    Xr,yr=robust(M['baseline'],X_src_tr,y_src_tr); M['adv']=make_xgb(seed).fit(Xr,yr)
    Xr2,yr2=robust(M['fewshot'],Xfs,yfs); M['fewshot_adv']=make_xgb(seed).fit(Xr2,yr2)
    return M

def eval_dir(direction,M,mean,scale,Xt,yt,seed):
    rng=np.random.RandomState(seed); n=min(MAXN,len(Xt)); idx=rng.choice(len(Xt),n,replace=False)
    Xt,yt=Xt[idx],yt[idx]; rows=[]
    for v,m in M.items():
        yp_clean=(m.predict_proba(Xt)[:,1]>=0.5).astype(int)
        base={'direction':direction,'seed':seed,'model':v,'condition':'clean'}; base.update(cm_metrics(yt,yp_clean))
        base['asr']=float('nan'); rows.append(base)
        S=saliency(m,Xt,yt)
        for e in EPS_EVAL:
            Xf=fgsm_functional(Xt,S,e,mean,scale); ypf=(m.predict_proba(Xf)[:,1]>=0.5).astype(int)
            r={'direction':direction,'seed':seed,'model':v,'condition':f'fgsm_eps{e}'}; r.update(cm_metrics(yt,ypf)); r['asr']=asr(yt,yp_clean,ypf); rows.append(r)
            Xp=pgd_functional(m,Xt,yt,e,mean,scale); ypp=(m.predict_proba(Xp)[:,1]>=0.5).astype(int)
            r2={'direction':direction,'seed':seed,'model':v,'condition':f'pgd_eps{e}'}; r2.update(cm_metrics(yt,ypp)); r2['asr']=asr(yt,yp_clean,ypp); rows.append(r2)
    return rows

ALL=[]
for seed in SEEDS:
    Xc_tr_raw,Xc_te_raw,yc_tr,yc_te=train_test_split(Xc_all,y_cic,test_size=0.3,random_state=seed,stratify=y_cic)
    scc=StandardScaler().fit(Xc_tr_raw); Xc_tr=scc.transform(Xc_tr_raw); Xc_te=scc.transform(Xc_te_raw)
    scu=StandardScaler().fit(Xu_tr_raw); Xu_tr=scu.transform(Xu_tr_raw); Xu_te=scu.transform(Xu_te_raw)
    Mc=train_variants('CIC->UNSW',Xc_tr,yc_tr,Xu_tr,y_utr,seed)
    ALL+=eval_dir('CIC->UNSW',Mc,scu.mean_,scu.scale_,Xu_te,y_ute,seed)
    Mu=train_variants('UNSW->CIC',Xu_tr,y_utr,Xc_tr,yc_tr,seed)
    ALL+=eval_dir('UNSW->CIC',Mu,scc.mean_,scc.scale_,Xc_te,yc_te,seed)
    print('seed',seed,'selesai')
df=pd.DataFrame(ALL); df.to_csv(os.path.join(OUTDIR,'reviewer_raw.csv'),index=False)
print('total rows',len(df))

## 4. Agregasi A2 (mean/std/95% CI) + tabel B5 + A5(ASR)

In [ ]:
def ci95(x):
    x=np.asarray(x,float); x=x[np.isfinite(x)]
    if len(x)<2: return (float('nan'),float('nan'))
    m=x.mean(); h=stats.t.ppf(0.975,len(x)-1)*x.std(ddof=1)/np.sqrt(len(x)); return (m-h,m+h)
agg=df.groupby(['direction','model','condition']).agg(
    mcc_mean=('mcc','mean'),mcc_std=('mcc','std'),
    asr_mean=('asr','mean'),recall_mean=('recall','mean'),precision_mean=('precision','mean'),
    f1_mean=('f1','mean')).reset_index()
ci=df.groupby(['direction','model','condition'])['mcc'].apply(lambda s: ci95(s.values)).reset_index(name='mcc_ci95')
agg=agg.merge(ci,on=['direction','model','condition'])
agg.to_csv(os.path.join(OUTDIR,'reviewer_agg.csv'),index=False)
import IPython.display as ipd; ipd.display(agg.round(4))
json.dump(agg.round(4).to_dict(orient='records'),open(os.path.join(OUTDIR,'reviewer_agg.json'),'w'),indent=2)
print('=== agregasi selesai (mean/std/CI, ASR, cm-metrics) ===')

## 5. Tabel multi-attack untuk paper (tab:multiattack) — FGSM vs PGD x epsilon

Membuat pivot MCC rerata-5-seed untuk FGSM 1-langkah vs PGD iteratif pada eps={0.05,0.1,0.2},
lalu **meng-generate baris LaTeX** yang tinggal ditempel ke `paper2-adversarial.tex` (Tabel `tab:multiattack`).
Angka NYATA dari eksperimen; tidak ada yang dikarang.

In [ ]:
# --- Pivot MCC: FGSM vs PGD x eps, rerata atas seed ---
EPS=[0.05,0.1,0.2]
DIRS=[('CIC->UNSW','CIC$\\rightarrow$UNSW'),('UNSW->CIC','UNSW$\\rightarrow$CIC')]
MODELS=[('baseline','baseline'),('fewshot','few-shot'),('adv','adv'),('fewshot_adv','\\textbf{few-shot+adv}')]
piv=(df[df['condition'].str.startswith(('fgsm_eps','pgd_eps'))]
     .groupby(['direction','model','condition'])['mcc'].mean().reset_index())
def getmcc(direction,model,attack,e):
    sub=piv[(piv['direction']==direction)&(piv['model']==model)&(piv['condition']==f'{attack}_eps{e}')]
    return float(sub['mcc'].iloc[0]) if len(sub) else float('nan')
def fmt(v):
    if not np.isfinite(v): return '--'
    s=f'{v:.3f}'.replace('.',',')            # koma desimal ala paper
    return ('$'+s+'$') if v>=0 else ('$'+s+'$')
# CSV wide untuk arsip
recs=[]
for dkey,_ in DIRS:
    for mkey,_ in MODELS:
        row={'direction':dkey,'model':mkey}
        for atk in ['fgsm','pgd']:
            for e in EPS: row[f'{atk}_mcc_eps{e}']=getmcc(dkey,mkey,atk,e)
        recs.append(row)
wide=pd.DataFrame(recs); wide.to_csv(os.path.join(OUTDIR,'multiattack_mcc.csv'),index=False)
import IPython.display as ipd; ipd.display(wide.round(4))
# --- Generate baris LaTeX persis format tab:multiattack ---
lines=[]
for dkey,dlbl in DIRS:
    lines.append(f'\\multirow{{4}}{{*}}{{{dlbl}}}')
    for i,(mkey,mlbl) in enumerate(MODELS):
        cells=[fmt(getmcc(dkey,mkey,'fgsm',e)) for e in EPS]+[fmt(getmcc(dkey,mkey,'pgd',e)) for e in EPS]
        lead=' & ' if i>0 else ' & '
        lines.append(f'{lead}{mlbl:<22} & '+' & '.join(cells)+' \\\\')
    lines.append('\\midrule' if dkey!=DIRS[-1][0] else '')
latex='\n'.join(l for l in lines if l!='')
open(os.path.join(OUTDIR,'multiattack_rows.tex'),'w').write(latex)
print('=== TEMPEL baris berikut ke Tabel tab:multiattack (ganti blok "--") ===\n')
print(latex)
print('\n=== file: multiattack_mcc.csv + multiattack_rows.tex di',OUTDIR,'===')

## 6. Uji signifikansi statistik (paired t-test + Wilcoxon) few-shot vs few-shot+adv

Menjawab review #8: apakah selisih antar-varian bermakna atau efek seed?
Uji berpasangan atas 5 seed (pasangan seed sama) per (direction, condition).

In [ ]:
from scipy.stats import ttest_rel, wilcoxon
def paired(direction,condition,mA='fewshot',mB='fewshot_adv'):
    a=df[(df['direction']==direction)&(df['model']==mA)&(df['condition']==condition)].sort_values('seed')['mcc'].values
    b=df[(df['direction']==direction)&(df['model']==mB)&(df['condition']==condition)].sort_values('seed')['mcc'].values
    n=min(len(a),len(b)); a,b=a[:n],b[:n]
    if n<2 or np.allclose(a,b): return dict(n=n,diff=float(np.mean(b-a)),t_p=float('nan'),w_p=float('nan'))
    try: tp=float(ttest_rel(b,a).pvalue)
    except Exception: tp=float('nan')
    try: wp=float(wilcoxon(b,a).pvalue)
    except Exception: wp=float('nan')
    return dict(n=n,diff=float(np.mean(b-a)),t_p=tp,w_p=wp)
sig_rows=[]
for direction in ['CIC->UNSW','UNSW->CIC']:
    for cond in ['clean','fgsm_eps0.1','pgd_eps0.1']:
        r={'direction':direction,'condition':cond}; r.update(paired(direction,cond)); sig_rows.append(r)
sig=pd.DataFrame(sig_rows); sig['significant_0.05']=sig['t_p']<0.05
sig.to_csv(os.path.join(OUTDIR,'significance_fewshot_vs_adv.csv'),index=False)
import IPython.display as ipd; ipd.display(sig.round(4))
print('=== paired t-test/Wilcoxon (fewshot vs fewshot+adv) selesai; diff=mean(B-A) ===')

## 7. Metrik perturbasi & validitas (review #10)

Metrik tambahan di luar MCC/ASR: magnitudo perturbasi (L-inf & L2 di z-space),
jumlah fitur termodifikasi, valid-flow rate & constraint-violation rate pasca proyeksi PCFS,
plus FPR & balanced accuracy. Dihitung utk fewshot & fewshot+adv per arah (seed pertama).

In [ ]:
def pcfs_valid(Xo):
    # Xo: ruang fitur ASLI. Valid bila memenuhi batasan PCFS aritmetik.
    ok=np.ones(len(Xo),bool)
    ok&=(Xo>=0).all(1)
    ok&=np.isclose(Xo[:,IX['fwd_pkts']],np.round(Xo[:,IX['fwd_pkts']]))
    ok&=np.isclose(Xo[:,IX['bwd_pkts']],np.round(Xo[:,IX['bwd_pkts']]))
    ok&=(Xo[:,IX['fwd_bytes']]>=Xo[:,IX['fwd_pkts']])
    ok&=(Xo[:,IX['bwd_bytes']]>=Xo[:,IX['bwd_pkts']])
    return ok
def perturb_metrics(model,Xt,yt,mean,scale,e=0.1):
    S=saliency(model,Xt,yt); Xf=fgsm_functional(Xt,S,e,mean,scale)
    d=Xf-Xt                                   # perturbasi di z-space
    linf=np.abs(d).max(1); l2=np.sqrt((d**2).sum(1))
    nmod=(np.abs(d)>1e-6).sum(1)
    Xo=Xf*scale+mean; valid=pcfs_valid(Xo)
    return dict(linf_mean=float(linf.mean()),l2_mean=float(l2.mean()),
                nmod_mean=float(nmod.mean()),valid_flow_rate=float(valid.mean()),
                constraint_violation_rate=float(1-valid.mean()))
def fpr_bal(y,yp):
    tn,fp,fn,tp=confusion_matrix(y,yp,labels=[0,1]).ravel()
    fpr=fp/(fp+tn) if (fp+tn)>0 else float('nan')
    tpr=tp/(tp+fn) if (tp+fn)>0 else float('nan')
    tnr=tn/(tn+fp) if (tn+fp)>0 else float('nan')
    return dict(fpr=float(fpr),balanced_acc=float((tpr+tnr)/2))
seed0=SEEDS[0]
Xc_tr_raw,Xc_te_raw,yc_tr,yc_te=train_test_split(Xc_all,y_cic,test_size=0.3,random_state=seed0,stratify=y_cic)
scc=StandardScaler().fit(Xc_tr_raw); scu=StandardScaler().fit(Xu_tr_raw)
Xc_tr=scc.transform(Xc_tr_raw); Xc_te=scc.transform(Xc_te_raw); Xu_tr=scu.transform(Xu_tr_raw); Xu_te=scu.transform(Xu_te_raw)
prows=[]
for direction,(Xsrc,ysrc,Xtgt_tr,ytgt_tr,Xte,yte,mean,scale) in {
  'CIC->UNSW':(Xc_tr,yc_tr,Xu_tr,y_utr,Xu_te,y_ute,scu.mean_,scu.scale_),
  'UNSW->CIC':(Xu_tr,y_utr,Xc_tr,yc_tr,Xc_te,yc_te,scc.mean_,scc.scale_)}.items():
    Mv=train_variants(direction,Xsrc,ysrc,Xtgt_tr,ytgt_tr,seed0)
    for v in ['fewshot','fewshot_adv']:
        m=Mv[v]; row={'direction':direction,'model':v}; row.update(perturb_metrics(m,Xte,yte,mean,scale))
        row.update(fpr_bal(yte,m.predict(Xte))); prows.append(row)
pm=pd.DataFrame(prows); pm.to_csv(os.path.join(OUTDIR,'perturbation_metrics.csv'),index=False)
import IPython.display as ipd; ipd.display(pm.round(4))
print('=== metrik perturbasi/validitas (review #10) selesai ===')

In [ ]:
# Upload artefak ke S3
try:
    import boto3; s3=boto3.client('s3',region_name=REGION); up=0
    for fn in sorted(os.listdir(OUTDIR)):
        if fn.endswith(('.csv','.json','.tex')): s3.upload_file(os.path.join(OUTDIR,fn),S3_BUCKET,f'unsw-far/paper2_reviewer/{fn}'); up+=1
    print('upload',up,'-> s3://%s/unsw-far/paper2_reviewer/'%S3_BUCKET)
except Exception as e: print('upload gagal:',e)